# 04. Audit Sinks — Tamper-evident logging

Every security-relevant action in Arc emits an `AuditEvent`. The model called a tool. The policy pipeline returned DENY. A child identity was derived. A signature failed to verify. Each of these is a single immutable event with the same shape, written to one or more pluggable sinks at the moment it happens.

This notebook walks through the audit module of `arctrust` — the canonical implementation of the **Audit** pillar from ADR-019. It is mock-first, runs without an API key, and writes everything into `tmp_path`-style temp directories.

**You will learn:**

- Why audit is a universal Arc primitive, not a federal-mode feature
- How `AuditEvent` is shaped — required fields, optional metadata, frozen-ness
- How `emit()` dispatches to a sink and *never* lets a sink failure break the caller
- How `WormSink` — the single durable, append-only, Ed25519-signed hash-chained audit
  log — combines what used to be two separate sinks (a plain JSONL writer and an
  in-memory-only signed chain) into one restart-safe compliance system of record
- How to write a custom `AuditSink` (it's just one method)
- How to fan a single `emit()` out to multiple sinks at once


## 1. Setup

Run the boilerplate below once. It walks up to the Arc repo root and adds every `packages/<pkg>/src` and `packages/<pkg>/tests` directory to `sys.path`. After this cell, `import arctrust` works directly from the source tree.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Confirm the audit surface is importable. We'll lean on every name listed below as the notebook progresses.

In [2]:
from arctrust import (
    AuditEvent,
    AuditSink,
    NullSink,
    WormSink,
    emit,
    generate_keypair,
    read_verified_anchor,
    verify_chain,
)
from arctrust.signer import ED25519, InProcessSigner

print("AuditEvent       :", AuditEvent)
print("AuditSink        :", AuditSink)
print("WormSink         :", WormSink)
print("NullSink         :", NullSink)
print("emit             :", emit)
print("verify_chain     :", verify_chain)
print("read_verified_anchor:", read_verified_anchor)

AuditEvent       : <class 'arctrust.audit.AuditEvent'>
AuditSink        : <class 'arctrust.audit.AuditSink'>
WormSink         : <class 'arctrust.audit.WormSink'>
NullSink         : <class 'arctrust.audit.NullSink'>
emit             : <function emit at 0x108f3c5e0>
verify_chain     : <function verify_chain at 0x108f3c4a0>
read_verified_anchor: <function read_verified_anchor at 0x108f3c540>


## 2. Why audit — the federal compliance lens

Audit is one of the **four pillars** every Arc deployment enforces (ADR-019):

1. **Identity** — every entity has a DID (covered in `01-identity-did`)
2. **Sign** — every loaded artifact is verified before use (`02-keypairs-signing`)
3. **Authorize** — every tool call passes through a policy pipeline (`03-policy-pipeline`)
4. **Audit** — every operation is recorded as an immutable event (this notebook)

Tier (personal / enterprise / federal) is *stringency metadata*, not a gate. Federal demands FIPS-validated crypto and signed allowlists; personal allows a self-signed bundle. **Every tier still audits.**

The compliance hooks are concrete:

| NIST 800-53 control | What it requires | Where arctrust covers it |
|---|---|---|
| **AU-2** Event types | Capture security-relevant events with actor, action, outcome | `AuditEvent` schema |
| **AU-5** Failure response | Audit failure must not interrupt the audited operation | `emit()` swallows sink errors |
| **AU-9** Protection | Detect tampering of stored audit records | `WormSink` Ed25519 hash chain |
| **AU-10** Non-repudiation | Every record is attributable and provably unforged | `WormSink` per-record Ed25519/ECDSA signature |
| **AU-11** Retention | Append-only, immutable, restart-safe storage | `WormSink` append-mode + tip recovery on restart |

FedRAMP and CMMC inherit these AU controls. The same code base runs in a developer's home directory and on a DOE machine in a SCIF — only the sinks and the operator key change.

## 3. The `AuditEvent` schema — what gets recorded

An `AuditEvent` is a frozen Pydantic model. Once constructed, it cannot be mutated — that's the whole point. Audit data must be a record of what happened, not a buffer that something later edits.

Required fields capture the universal who/what/when/outcome:

In [3]:
evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="tool.call",
    target="read_file",
    outcome="allow",
)

print("actor_did :", evt.actor_did)
print("action    :", evt.action)
print("target    :", evt.target)
print("outcome   :", evt.outcome)
print("ts        :", evt.ts)

actor_did : did:arc:walkthrough:exec/aabbccdd
action    : tool.call
target    : read_file
outcome   : allow
ts        : 2026-07-09T11:39:17.917851+00:00


Notice the `ts` field auto-populates with an ISO 8601 UTC timestamp at construction time. We never trust the caller to pass a timestamp — the audit module owns that.

Optional fields cover request correlation, tier metadata, classification, and tamper-evidence of the request payload itself (a SHA-256, never raw data):

In [4]:
import hashlib

payload = b'{"path": "/etc/passwd"}'
evt_full = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="tool.call",
    target="read_file",
    outcome="deny",
    classification="UNCLASSIFIED",
    tier="federal",
    request_id="req-2026-05-07-0001",
    payload_hash=hashlib.sha256(payload).hexdigest(),
    extra={"reason": "path not on allowlist"},
)

for k, v in evt_full.model_dump().items():
    print(f"  {k:<14}: {v}")

  actor_did     : did:arc:walkthrough:exec/aabbccdd
  action        : tool.call
  target        : read_file
  outcome       : deny
  classification: UNCLASSIFIED
  tier          : federal
  request_id    : req-2026-05-07-0001
  payload_hash  : 379f257c1f25d1ac9afd683b4c6b72dc3896ff9b2e5602a20aa94d584af96ad8
  ts            : 2026-07-09T11:39:17.920773+00:00
  extra         : {'reason': 'path not on allowlist'}


The frozen-ness is not advisory. Pydantic enforces it — any attempt to mutate a field on an event after creation raises:

In [5]:
try:
    evt.outcome = "allow"  # type: ignore[misc]
except Exception as e:
    print("raised:", type(e).__name__, str(e).splitlines()[0])

raised: ValidationError 1 validation error for AuditEvent


Because the model is fully serialisable, every event round-trips through JSON without losing information — the foundation for every sink we'll see next.

In [6]:
import json

as_json = json.dumps(evt_full.model_dump(), default=str, indent=2)
print(as_json)

{
  "actor_did": "did:arc:walkthrough:exec/aabbccdd",
  "action": "tool.call",
  "target": "read_file",
  "outcome": "deny",
  "classification": "UNCLASSIFIED",
  "tier": "federal",
  "request_id": "req-2026-05-07-0001",
  "payload_hash": "379f257c1f25d1ac9afd683b4c6b72dc3896ff9b2e5602a20aa94d584af96ad8",
  "ts": "2026-07-09T11:39:17.920773+00:00",
  "extra": {
    "reason": "path not on allowlist"
  }
}


## 4. The `emit()` function — single point of dispatch

Every part of Arc that needs to audit something does the same thing: build an `AuditEvent`, then call `emit(event, sink)`. That's it. The emit function is *deliberately* tiny:

```python
def emit(event: AuditEvent, sink: AuditSink) -> None:
    try:
        sink.write(event)
    except Exception:
        _logger.warning(...)  # AU-5: never propagate
```

The contract is: **the audit subsystem will never break the operation it is auditing**. If the disk is full, if the network sink is down, if a custom sink throws — `emit` swallows it, logs at WARNING, and returns. That's NIST AU-5 (Response to Audit Processing Failures).

Let's see this in action with a deliberately broken sink:

In [7]:
class ExplodingSink:
    """A sink that raises on every write — to demonstrate AU-5 behaviour."""

    def write(self, event: AuditEvent) -> None:
        raise RuntimeError("disk on fire")


evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="emit.demo",
    target="broken-sink",
    outcome="allow",
)

# emit() must NOT raise. The caller's path is preserved.
emit(evt, ExplodingSink())  # type: ignore[arg-type]
print("caller still alive — emit() ate the RuntimeError, exactly per AU-5")

Audit sink 'ExplodingSink' raised on event 'emit.demo' — swallowing (AU-5)
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-a7b1b963ad73e08f9/packages/arctrust/src/arctrust/audit.py", line 506, in emit
    sink.write(event)
  File "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_28438/1093638037.py", line 5, in write
    raise RuntimeError("disk on fire")
RuntimeError: disk on fire


caller still alive — emit() ate the RuntimeError, exactly per AU-5


If we had not used `emit()` and called `sink.write()` directly, the runtime would have crashed and the audited operation would have failed. That's why arctrust never recommends calling sinks directly — `emit()` is the safe API.

## 5. `WormSink` — durable, append-only, Ed25519-signed hash chain

`WormSink` is the single durable compliance sink. It used to be two separate things — a
plain append-only `JsonlSink` and an in-memory-only `SignedChainSink` — but neither alone
was enough: a plain JSONL file is honest about not detecting tampering, and an in-memory
chain doesn't survive a process restart. `WormSink` merges both properties into one file:

- **Append-only at the OS level.** Opened `O_APPEND`, mode `0600`. That's NIST AU-11.
- **Hash-chained.** Every record's `event_hash` commits `(seq, prev_hash, event)`, so
  tampering with record N breaks record N+1's link.
- **Ed25519 (or ECDSA-P256 under FIPS) signed.** Each record's `event_hash` is signed by
  an operator `Signer` — NIST AU-10 non-repudiation.
- **Restart-safe.** The chain tip and next `seq` are recovered from the file tail on
  construction, unlike the old in-memory-only chain.
- **Single-writer.** An exclusive `flock` is held for the sink's lifetime.

It needs a real `Signer` — the same primitive `arcllm`'s asymmetric request-signing uses
(see `arcllm/12-security-layer.ipynb`). We'll mint one from a fresh Ed25519 keypair:

In [8]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp(prefix="arctrust-audit-"))
log_path = tmp / "audit.jsonl"
print("log_path:", log_path)

log_path: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arctrust-audit-fdpwmz_b/audit.jsonl


Generate an operator keypair, wrap it in a `Signer`, and construct a `WormSink` pointing at the path:

In [9]:
operator = generate_keypair()
signer = InProcessSigner(operator.private_key, ED25519)
worm = WormSink(log_path, signer)

print("chain tip before any writes:", repr(worm.chain_tip))

for i in range(4):
    evt = AuditEvent(
        actor_did=f"did:arc:walkthrough:exec/{i:08x}",
        action="tool.call",
        target="read_file",
        outcome="allow" if i % 2 == 0 else "deny",
        request_id=f"req-{i:04d}",
    )
    emit(evt, worm)

print("chain tip after 4 writes:", worm.chain_tip[:24], "…")
print("file size:", log_path.stat().st_size, "bytes")
print("line count:", len(log_path.read_text().splitlines()))

chain tip before any writes: ''
chain tip after 4 writes: 9b328419503211dc940be1b6 …
file size: 2322 bytes
line count: 4


Read records back as JSON lines. Each line is now `{seq, event, prev_hash, event_hash, algorithm, signature}` — the compliance archive and the tamper-evidence chain are the same file:

In [10]:
import json

for raw in log_path.read_text().splitlines():
    rec = json.loads(raw)
    evt_data = rec["event"]
    print(
        f"  seq={rec['seq']}  {evt_data['ts']}  {evt_data['action']:<10}  "
        f"{evt_data['target']:<10}  {evt_data['outcome']}  ({evt_data['actor_did']})"
    )

  seq=0  2026-07-09T11:39:17.935644+00:00  tool.call   read_file   allow  (did:arc:walkthrough:exec/00000000)
  seq=1  2026-07-09T11:39:17.935987+00:00  tool.call   read_file   deny  (did:arc:walkthrough:exec/00000001)
  seq=2  2026-07-09T11:39:17.936278+00:00  tool.call   read_file   allow  (did:arc:walkthrough:exec/00000002)
  seq=3  2026-07-09T11:39:17.936554+00:00  tool.call   read_file   deny  (did:arc:walkthrough:exec/00000003)


Two important properties of `WormSink` worth calling out explicitly:

1. **Append-only at the OS level.** The file is opened `O_APPEND`, mode `0600`; the constructor never truncates. Restarts recover the tip from the file tail.
2. **Every record is chained AND signed.** Unlike the old `JsonlSink`, there's no unsigned/unchained mode — durability and tamper-evidence are the same write.

Inspect the structure of one record directly from the file. The combination of `prev_hash`, `event_hash`, and `signature` is what makes the chain tamper-evident:

In [11]:
lines = log_path.read_text().splitlines()
rec = json.loads(lines[2])

print("record keys:", list(rec.keys()))
print("seq        :", rec["seq"])
print("prev_hash  :", rec["prev_hash"][:24], "…")
print("event_hash :", rec["event_hash"][:24], "…")
print("algorithm  :", rec["algorithm"])
print(
    "signature  :",
    rec["signature"][:24],
    "…",
    "(" + str(len(bytes.fromhex(rec["signature"]))) + " bytes)",
)
print("event keys :", list(rec["event"].keys()))

record keys: ['seq', 'event', 'prev_hash', 'event_hash', 'algorithm', 'signature']
seq        : 2
prev_hash  : 60d8c362584a7b8095e40e9e …
event_hash : 3390c072f0d92c0885966825 …
algorithm  : ed25519
signature  : 31d3632f74ce83f43d2ec8ea … (64 bytes)
event keys : ['actor_did', 'action', 'target', 'outcome', 'classification', 'tier', 'request_id', 'payload_hash', 'ts', 'extra']


## 6. Verifying the chain

There are two ways to check a chain's integrity:

- **`worm.verify_chain()`** — the instance method. Uses the sink's own cached public key.
- **The standalone `verify_chain(path, public_key)`** — a lock-free, streaming, read-only
  function. This is what `arc store verify` and offline auditors use: it never needs to
  hold the sink's exclusive `flock`, so it works against a live, still-being-written chain,
  or from a completely separate process that only has the operator's public key.

Both walk every record from genesis, recomputing `event_hash` from `(seq, prev_hash, event)`,
checking the Ed25519/ECDSA signature, and confirming `seq` is contiguous with no gaps.

In [12]:
print("worm.verify_chain()                     ->", worm.verify_chain())
print("verify_chain(path, operator.public_key) ->", verify_chain(log_path, operator.public_key))

worm.verify_chain()                     -> True
verify_chain(path, operator.public_key) -> True


## 7. Tampering demo — what a chain breach looks like

Now we play attacker with direct filesystem access. We'll edit one stored record's event
content on disk and observe that `verify_chain()` returns `False`. `WormSink` holds an
advisory `flock` on the active file, not a mandatory OS-level lock — a second process (or a
plain `open("wb")` from this same notebook, bypassing `flock`) can still rewrite the bytes.
That's exactly the threat model `verify_chain()` defends against.

First, capture the file's original bytes and pick a record to mutate:

In [13]:
original_bytes = log_path.read_bytes()
original_lines = original_bytes.splitlines()
target_index = 1
original_record = json.loads(original_lines[target_index])
print("target record action (before):", original_record["event"]["action"])
print("target record outcome (before):", original_record["event"]["outcome"])

target record action (before): tool.call
target record outcome (before): deny


Mutate the event content in place, then rewrite the file with the tampered line spliced in. Imagine an attacker rewriting a `deny` to an `allow` to hide the fact that they were rejected by policy — the `event_hash` and `signature` are left untouched, exactly as an attacker who doesn't hold the operator's private key would have to leave them:

In [14]:
tampered_event = {**original_record["event"], "outcome": "allow", "action": "tampered_action"}
tampered_record = {**original_record, "event": tampered_event}

tampered_lines = list(original_lines)
tampered_lines[target_index] = json.dumps(tampered_record).encode("utf-8")
log_path.write_bytes(b"\n".join(tampered_lines) + b"\n")

reread = json.loads(log_path.read_text().splitlines()[target_index])
print("target record action (after) :", reread["event"]["action"])
print("target record outcome (after):", reread["event"]["outcome"])

target record action (after) : tampered_action
target record outcome (after): allow


Run verification — using the lock-free standalone `verify_chain(path, public_key)`, since
`worm` (the in-process object) is oblivious to a rewrite that happened underneath it via a
separate file handle. We wrap the call in a small helper that reports the failure clearly,
the way an audit-monitoring tool would:

In [15]:
def verify_or_alert(path: Path, public_key: bytes, label: str) -> None:
    try:
        ok = verify_chain(path, public_key)
        if not ok:
            print(f"tamper detected: chain integrity check failed for {label!r}")
        else:
            print(f"chain intact for {label!r}")
    except Exception as exc:
        print(f"tamper detected: {label!r} raised {type(exc).__name__}: {exc}")


verify_or_alert(log_path, operator.public_key, "after-tamper")

tamper detected: chain integrity check failed for 'after-tamper'


The chain has been broken. The hash mismatch at the tampered record means every downstream
record's `prev_hash` lineage no longer fits — even if the attacker re-signed only the
tampered record, the signature was over the original `event_hash`, not the new one. To
produce a valid chain the attacker would need the operator's private key, which never
touches this file (it's held by the `Signer`, out-of-band from the WORM chain itself).

This is what tamper-evident logging means in practice: the moment somebody edits the audit
trail — even bypassing `WormSink`'s own `flock` entirely — `verify_chain()` sees it.

Restore the file and confirm verification passes again. (In real systems you wouldn't
restore — you'd alert and quarantine the log file; we restore here so the rest of the
notebook continues from a clean state.)

In [16]:
log_path.write_bytes(original_bytes)
verify_or_alert(log_path, operator.public_key, "restored")
worm.close()  # release the flock now that we're done writing through `worm`

chain intact for 'restored'


## 8. Writing a custom `AuditSink`

`AuditSink` is a runtime-checkable Protocol — *any* object with a `write(event)` method satisfies it. No inheritance, no mixin, no registration step. This is deliberate: every deployment has its own infrastructure (Splunk, Elastic, CloudWatch, syslog, NATS, plain stdout, an in-memory buffer for tests) and the audit module shouldn't care which.

Here are three custom sinks in under 30 lines combined:

In [17]:
class InMemorySink:
    """Capture events into a list. Useful for tests and assertions."""

    def __init__(self) -> None:
        self.events: list[AuditEvent] = []

    def write(self, event: AuditEvent) -> None:
        self.events.append(event)


class CountingSink:
    """Count events by (action, outcome). The shape of a metrics emitter."""

    def __init__(self) -> None:
        self.counts: dict[tuple[str, str], int] = {}

    def write(self, event: AuditEvent) -> None:
        key = (event.action, event.outcome)
        self.counts[key] = self.counts.get(key, 0) + 1


class StdoutSink:
    """Print a one-line human-readable summary. Useful in dev."""

    def write(self, event: AuditEvent) -> None:
        print(f"[{event.ts}] {event.actor_did} {event.action}({event.target}) -> {event.outcome}")

All three pass the `isinstance(sink, AuditSink)` Protocol check at runtime — no inheritance required:

In [18]:
in_mem = InMemorySink()
counts = CountingSink()
stdout = StdoutSink()

for sink in (in_mem, counts, stdout):
    print(f"{type(sink).__name__:<14} isinstance(AuditSink) -> {isinstance(sink, AuditSink)}")

InMemorySink   isinstance(AuditSink) -> True
CountingSink   isinstance(AuditSink) -> True
StdoutSink     isinstance(AuditSink) -> True


Drive each sink through `emit()` and confirm they behave as expected:

In [19]:
for action, outcome in [
    ("tool.call", "allow"),
    ("tool.call", "deny"),
    ("policy.evaluate", "allow"),
    ("policy.evaluate", "deny"),
    ("tool.call", "allow"),
]:
    evt = AuditEvent(
        actor_did="did:arc:walkthrough:exec/aabbccdd",
        action=action,
        target="some_tool",
        outcome=outcome,
    )
    emit(evt, in_mem)
    emit(evt, counts)

print("InMemorySink   ->", len(in_mem.events), "events captured")
print("CountingSink   ->")
for (action, outcome), n in counts.counts.items():
    print(f"    ({action:<18}, {outcome:<5}): {n}")

InMemorySink   -> 5 events captured
CountingSink   ->
    (tool.call         , allow): 2
    (tool.call         , deny ): 1
    (policy.evaluate   , allow): 1
    (policy.evaluate   , deny ): 1


`NullSink` ships in `arctrust` for the special case where you genuinely want to discard every event — typically test fixtures or air-gapped evaluation harnesses where you don't want side effects. It satisfies the Protocol with a no-op `write`:

In [20]:
null = NullSink()
evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="tool.call",
    target="black_hole",
    outcome="allow",
)
emit(evt, null)
print("NullSink records (always empty):", null.records)

NullSink records (always empty): []


## 9. Multi-sink fan-out — the production pattern

Real Arc deployments rarely use a single sink. The typical wiring fans every event out to:

- A **`WormSink`** for the durable, tamper-evident compliance record (AU-9/AU-10/AU-11 in
  one file — no separate unsigned-archive sink needed anymore)
- A **metrics sink** for live observability (counts by action/outcome)
- An **in-memory sink** for tests / assertions

There's no `MultiSink` class because there doesn't need to be. A three-line tuple plus a loop is the entire pattern, and `emit()` already handles per-sink failures:

In [21]:
import tempfile
from pathlib import Path

fanout_dir = Path(tempfile.mkdtemp(prefix="arctrust-fanout-"))
fanout_signer = InProcessSigner(generate_keypair().private_key, ED25519)
fanout_worm = WormSink(fanout_dir / "audit.worm", fanout_signer)
metrics = CountingSink()
memory = InMemorySink()

all_sinks: tuple[AuditSink, ...] = (fanout_worm, metrics, memory)


def fan_out(event: AuditEvent, sinks: tuple[AuditSink, ...]) -> None:
    for s in sinks:
        emit(event, s)  # emit() isolates per-sink failures

Drive a small workflow's worth of audit events through the fan-out — the kind of trace a single agent run would produce:

In [22]:
workflow_events = [
    ("agent.start", "agent_loop", "allow"),
    ("policy.evaluate", "read_file", "allow"),
    ("tool.call", "read_file", "allow"),
    ("policy.evaluate", "write_file", "deny"),
    ("tool.call", "write_file", "deny"),
    ("agent.complete", "agent_loop", "allow"),
]

for action, target, outcome in workflow_events:
    evt = AuditEvent(
        actor_did="did:arc:walkthrough:exec/agent01",
        action=action,
        target=target,
        outcome=outcome,
        tier="federal",
        request_id="workflow-001",
    )
    fan_out(evt, all_sinks)

print("worm chain records   :", len((fanout_dir / "audit.worm").read_text().splitlines()))
print("worm chain verifies  :", fanout_worm.verify_chain())
print("metrics counts       :")
for k, v in metrics.counts.items():
    print(f"    {k}: {v}")
print("memory events        :", len(memory.events))

worm chain records   : 6
worm chain verifies  : True
metrics counts       :
    ('agent.start', 'allow'): 1
    ('policy.evaluate', 'allow'): 1
    ('tool.call', 'allow'): 1
    ('policy.evaluate', 'deny'): 1
    ('tool.call', 'deny'): 1
    ('agent.complete', 'allow'): 1
memory events        : 6


Every sink saw every event. The WORM chain is on disk and verifies, the metrics counter is up to date, and an in-memory copy is available for tests or live observability. One `fan_out()` call, three storage backends, zero coupling.

And critically — if any one sink had failed (full disk, network blip), `emit()` would have isolated the failure to that sink and the others would still have received the event. Let's prove it by mixing a working sink with the deliberately broken one:

In [23]:
resilient_sinks: tuple[AuditSink, ...] = (fanout_worm, ExplodingSink(), memory)  # type: ignore[arg-type]

evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/agent01",
    action="resilience.test",
    target="mixed-sinks",
    outcome="allow",
)
fan_out(evt, resilient_sinks)

print("worm chain record count:", len((fanout_dir / "audit.worm").read_text().splitlines()))
print("memory event count     :", len(memory.events))
print("caller still alive     : True")

Audit sink 'ExplodingSink' raised on event 'resilience.test' — swallowing (AU-5)
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-a7b1b963ad73e08f9/packages/arctrust/src/arctrust/audit.py", line 506, in emit
    sink.write(event)
  File "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_28438/1093638037.py", line 5, in write
    raise RuntimeError("disk on fire")
RuntimeError: disk on fire


worm chain record count: 7
memory event count     : 7
caller still alive     : True


The exploding sink raised, the warning was logged, the other two sinks still received the event, and the calling code carried on. That's AU-5 fan-out resilience in 30 lines.

## 10. Summary

**API surface covered:**

| Symbol | Role |
|---|---|
| `AuditEvent` | Frozen Pydantic schema for all security-relevant events |
| `AuditSink` | Runtime-checkable Protocol — any `write(event)` method qualifies |
| `WormSink` | Durable, append-only, Ed25519/ECDSA-signed hash-chained sink (NIST AU-9/AU-10/AU-11) — replaces the old `JsonlSink` + `SignedChainSink` pair |
| `verify_chain()` | Lock-free, streaming, read-only chain verification (instance method + standalone function) |
| `NullSink` | No-op sink for tests and air-gapped evaluation |
| `emit()` | Safe dispatch — swallows sink failures (NIST AU-5) |

**Patterns demonstrated:**

- Build an event, hand it to `emit(event, sink)` — never call sinks directly
- `WormSink` needs a real `Signer` (from `arctrust.signer`) and a real filesystem path — it's not a symmetric-secret or file-like-object sink
- Tampering with a stored record on disk (even bypassing the sink's own `flock`) breaks `verify_chain()`
- Custom sinks are a one-method protocol implementation; no inheritance, no registration
- Fan-out is a tuple plus a loop — there's no `MultiSink` because there doesn't need to be
- A failing sink in a fan-out does not block the other sinks or the caller

**Compliance hooks landed:**

- AU-2 (event capture) via `AuditEvent`
- AU-5 (failure response) via `emit()`
- AU-9 (protection of audit information) via `WormSink`'s hash chain
- AU-10 (non-repudiation) via `WormSink`'s per-record Ed25519/ECDSA signature
- AU-11 (retention) via `WormSink`'s append-mode + restart-safe tip recovery

**Where the audit story continues:**

- `walkthroughs/arcllm/10-audit-trail.ipynb` — `AuditModule` in the arcllm module stack,   emitting through the same `arctrust.audit.emit` we just used.
- `walkthroughs/arcllm/14-trace-store.ipynb` — Hash-chained `TraceStore` for full LLM call   traces with daily rotation and warm-start tamper detection.

Both build directly on the primitives in this notebook. The audit pillar starts here.